# Honest CatBoost comparison: safe features + train-only popularity + semantic features

Цель ноутбука - честно сравнить:

```text
1. safe features
2. safe features + train-only popularity
3. safe features + train-only popularity + semantic features
```

Важные ограничения:
- не используются target labels как признаки;
- не используются `user_id` и `target_app_id` как признаки;
- не используются global Steam aggregates вроде `positive`, `negative`, `recommendations`, `peak_ccu`;
- popularity-признаки берутся только из train-only файла `train_item_popularity.csv`;
- semantic features добавляются отдельным блоком.


In [ ]:
# Optional dependency installation:
# !pip install -q catboost pandas pyarrow scikit-learn


In [ ]:
import os, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("F:/Coursework")

print("PROJECT_ROOT:", PROJECT_ROOT)


## 1. Пути


In [ ]:
train_tab_path = PROJECT_ROOT / "data/final/tabular_temporal/train_tabular.parquet"
val_tab_path = PROJECT_ROOT / "data/final/tabular_temporal/val_tabular.parquet"
test_tab_path = PROJECT_ROOT / "data/final/tabular_temporal/test_tabular.parquet"

popularity_path = PROJECT_ROOT / "outputs/baselines/train_item_popularity.csv"
old_metrics_path = PROJECT_ROOT / "outputs/baselines/metrics_baselines.csv"

semantic_dir = PROJECT_ROOT / "outputs/semantic"
semantic_zip = semantic_dir / "steam_semantic_outputs.zip"

out_dir = PROJECT_ROOT / "outputs" / "semantic" / "honest_catboost"
out_dir.mkdir(parents=True, exist_ok=True)

for p in [train_tab_path, val_tab_path, test_tab_path, popularity_path]:
    print(p, "OK" if p.exists() else "MISSING")
print("semantic_dir:", semantic_dir, "exists:", semantic_dir.exists())
print("semantic_zip:", semantic_zip, "exists:", semantic_zip.exists())


## 2. Распаковка semantic outputs, если они лежат архивом


In [ ]:
expected_semantic_files = [
    "train_semantic_features.parquet",
    "val_semantic_features.parquet",
    "test_semantic_features.parquet",
]

if semantic_zip.exists():
    missing = [f for f in expected_semantic_files if not list(semantic_dir.rglob(f))]
    if missing:
        with zipfile.ZipFile(semantic_zip, "r") as z:
            z.extractall(semantic_dir)
        print("Semantic zip extracted")
    else:
        print("Semantic parquet files already exist")
else:
    print("No semantic zip found; expecting semantic parquet files in outputs/semantic")

print("semantic files:", [p.name for p in semantic_dir.rglob("*") if p.is_file()][:30])


## 3. Загрузка tabular и semantic


In [ ]:
def find_semantic_file(name):
    hits = list(semantic_dir.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Required semantic file was not found: {name}")
    return hits[0]

train_tab = pd.read_parquet(train_tab_path)
val_tab = pd.read_parquet(val_tab_path)
test_tab = pd.read_parquet(test_tab_path)

train_sem = pd.read_parquet(find_semantic_file("train_semantic_features.parquet"))
val_sem = pd.read_parquet(find_semantic_file("val_semantic_features.parquet"))
test_sem = pd.read_parquet(find_semantic_file("test_semantic_features.parquet"))

print("tabular shapes:", train_tab.shape, val_tab.shape, test_tab.shape)
print("semantic shapes:", train_sem.shape, val_sem.shape, test_sem.shape)
print("train_tab columns:", train_tab.columns.tolist())
print("train_sem columns:", train_sem.columns.tolist())


## 4. Merge semantic features


In [ ]:
SEM_COLS = [
    "semantic_target_liked_cosine",
    "semantic_target_disliked_cosine",
    "semantic_similarity_gap",
    "semantic_max_liked_cosine",
    "semantic_max_disliked_cosine",
    "semantic_top3_liked_mean_cosine",
    "semantic_top3_disliked_mean_cosine",
]

def merge_semantic(tab, sem):
    sem_cols = [c for c in SEM_COLS if c in sem.columns]
    if "sample_id" in tab.columns and "sample_id" in sem.columns:
        return tab.merge(sem[["sample_id"] + sem_cols], on="sample_id", how="left")
    res = tab.reset_index(drop=True).copy()
    sem2 = sem[sem_cols].reset_index(drop=True)
    if len(res) != len(sem2):
        raise ValueError("Semantic and tabular datasets have different number of rows; row-wise merge is not safe.")
    return pd.concat([res, sem2], axis=1)

train = merge_semantic(train_tab, train_sem)
val = merge_semantic(val_tab, val_sem)
test = merge_semantic(test_tab, test_sem)

print("merged shapes:", train.shape, val.shape, test.shape)
print("semantic NaN share:")
print(train[SEM_COLS].isna().mean())


## 5. Merge train-only popularity


In [ ]:
def infer_target_col(df):
    for c in ["target_app_id", "target_item_id", "app_id"]:
        if c in df.columns:
            return c
    raise ValueError("Target item id column was not found in tabular data.")

def infer_pop_app_col(df):
    for c in ["target_app_id", "app_id", "item_id", "steam_appid"]:
        if c in df.columns:
            return c
    raise ValueError("Item id column was not found in train_item_popularity.csv.")

target_col = infer_target_col(train)
print("target_col:", target_col)

popularity_df = pd.read_csv(popularity_path)
pop_app_col = infer_pop_app_col(popularity_df)
print("popularity columns:", popularity_df.columns.tolist())
print("pop_app_col:", pop_app_col)

# Оставляем только числовые train-only popularity признаки.
# Важно: этот файл был построен только на train split, поэтому эти признаки можно использовать в честном benchmark.
pop_numeric_cols = [
    c for c in popularity_df.columns
    if c != pop_app_col and pd.api.types.is_numeric_dtype(popularity_df[c])
]

pop_rename = {}
for c in pop_numeric_cols:
    if c.startswith("train_item_"):
        pop_rename[c] = c
    else:
        pop_rename[c] = f"train_item_{c}"

popularity_for_merge = popularity_df[[pop_app_col] + pop_numeric_cols].rename(columns={pop_app_col: target_col, **pop_rename})

# Удаляем уже существующие одноимённые popularity columns, чтобы merge был однозначным.
for df in [train, val, test]:
    duplicate_cols = [c for c in popularity_for_merge.columns if c != target_col and c in df.columns]
    if duplicate_cols:
        df.drop(columns=duplicate_cols, inplace=True)

train = train.merge(popularity_for_merge, on=target_col, how="left")
val = val.merge(popularity_for_merge, on=target_col, how="left")
test = test.merge(popularity_for_merge, on=target_col, how="left")

train_only_popularity_features = [
    c for c in popularity_for_merge.columns
    if c != target_col and c in train.columns and pd.api.types.is_numeric_dtype(train[c])
]

print("train-only popularity features:", train_only_popularity_features)
print("missing popularity share on test:")
print(test[train_only_popularity_features].isna().mean())


## 6. Labels and leakage-safe feature sets


In [ ]:
def get_label_col(df):
    for c in ["label_strict", "is_recommended", "label"]:
        if c in df.columns:
            return c
    raise ValueError("Label column was not found.")

def label_to_int(s):
    if s.dtype == bool:
        return s.astype(int)
    return s.astype(str).str.lower().isin(["1", "true", "yes", "recommended"]).astype(int)

label_col = get_label_col(train)
y_train = label_to_int(train[label_col])
y_val = label_to_int(val[label_col])
y_test = label_to_int(test[label_col])

SAFE_BASE_CANDIDATES = [
    "history_len",
    "history_positive_count",
    "history_negative_count",
    "history_positive_share",
    "history_total_hours",
    "history_mean_hours",
    "history_median_hours",
    "history_liked_mean_hours",
    "history_disliked_mean_hours",
    "target_liked_jaccard",
    "target_disliked_jaccard",
    "target_jaccard_diff",
    "target_liked_overlap_count",
    "target_disliked_overlap_count",
    "target_description_len",
    "target_title_len",
    "price",
    "required_age",
    "dlc_count",
    "release_year",
]

FORBIDDEN_EXACT = {
    "label_strict",
    "label_hybrid",
    "label_hours_relative",
    "is_recommended",
    "output",
    "target_hours",
    "target_hours_aux",
    "user_id",
    "target_app_id",
    "app_id",
    "review_id",
    "positive_ratio",
    "user_reviews",
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "average_playtime_forever",
    "median_playtime_forever",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent",
}

FORBIDDEN_PATTERNS = [
    "label",
    "output",
    "target_hours",
    "review_id",
    "hours_relative",
    "hours_reference",
    "low_hours_positive",
    "high_hours_negative",
]

def is_allowed_numeric_feature(c):
    cl = c.lower()
    if c in FORBIDDEN_EXACT:
        return False
    if any(p in cl for p in FORBIDDEN_PATTERNS):
        return False
    if not pd.api.types.is_numeric_dtype(train[c]):
        return False
    return True

safe_base_features = [
    c for c in SAFE_BASE_CANDIDATES
    if c in train.columns and is_allowed_numeric_feature(c)
]

popularity_features = [
    c for c in train_only_popularity_features
    if c in train.columns and is_allowed_numeric_feature(c)
]

semantic_features = [
    c for c in SEM_COLS
    if c in train.columns and is_allowed_numeric_feature(c)
]

safe_plus_popularity_features = safe_base_features + [c for c in popularity_features if c not in safe_base_features]
safe_plus_popularity_semantic_features = safe_plus_popularity_features + [
    c for c in semantic_features if c not in safe_plus_popularity_features
]

def validate_features(features, name):
    bad = [c for c in features if not is_allowed_numeric_feature(c)]
    if bad:
        raise ValueError(f"{name} contains forbidden or non-numeric features: {bad}")
    if len(features) == 0:
        raise ValueError(f"{name} is empty. Check the input column names and preprocessing pipeline.")

validate_features(safe_base_features, "safe_base_features")
validate_features(safe_plus_popularity_features, "safe_plus_popularity_features")
validate_features(safe_plus_popularity_semantic_features, "safe_plus_popularity_semantic_features")

print("label_col:", label_col)
print("safe_base_features:", len(safe_base_features), safe_base_features)
print("popularity_features:", len(popularity_features), popularity_features)
print("semantic_features:", len(semantic_features), semantic_features)
print("safe_plus_popularity_features:", len(safe_plus_popularity_features))
print("safe_plus_popularity_semantic_features:", len(safe_plus_popularity_semantic_features))


## 7. CatBoost training and evaluation


In [ ]:
def best_threshold_by_f1(y_true, scores):
    best = {"threshold": 0.5, "f1": -1}
    for thr in np.linspace(0.01, 0.99, 99):
        pred = (scores >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(thr), "precision": float(p), "recall": float(r), "f1": float(f1)}
    return best["threshold"]

def calc_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "threshold": float(threshold),
    }

def train_eval_catboost(name, features):
    X_train = train[features].copy()
    X_val = val[features].copy()
    X_test = test[features].copy()

    model = CatBoostClassifier(
        iterations=1200,
        learning_rate=0.04,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        verbose=100,
        early_stopping_rounds=100,
        allow_writing_files=False
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test)[:, 1]

    thr = best_threshold_by_f1(y_val, val_score)
    val_metrics = calc_metrics(y_val, val_score, thr)
    test_metrics = calc_metrics(y_test, test_score, thr)

    pred_val = pd.DataFrame({"y_true": y_val, "score": val_score, "pred": (val_score >= thr).astype(int)})
    pred_test = pd.DataFrame({"y_true": y_test, "score": test_score, "pred": (test_score >= thr).astype(int)})

    model.save_model(out_dir / f"{name}.cbm")
    pred_val.to_csv(out_dir / f"{name}_val_predictions.csv", index=False)
    pred_test.to_csv(out_dir / f"{name}_test_predictions.csv", index=False)

    fi = pd.DataFrame({"feature": features, "importance": model.get_feature_importance()})
    fi.sort_values("importance", ascending=False).to_csv(out_dir / f"{name}_feature_importance.csv", index=False)

    return [
        {"method": name, "split": "val", **val_metrics, "n_features": len(features)},
        {"method": name, "split": "test", **test_metrics, "n_features": len(features)},
    ]

rows = []
rows += train_eval_catboost("catboost_safe_base", safe_base_features)
rows += train_eval_catboost("catboost_safe_plus_train_popularity", safe_plus_popularity_features)
rows += train_eval_catboost("catboost_safe_plus_train_popularity_semantic", safe_plus_popularity_semantic_features)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(out_dir / "metrics_honest_catboost_semantic.csv", index=False)
metrics_df


## 8. Comparison with previously saved baseline metrics


In [ ]:
if old_metrics_path.exists():
    old_metrics = pd.read_csv(old_metrics_path)
    print("Previously saved baseline rows:")
    display(old_metrics[old_metrics.astype(str).apply(lambda row: row.str.contains("safe_catboost_plus_popularity", case=False, regex=False).any(), axis=1)])
else:
    print("Previous baseline metrics file was not found.")

pivot = metrics_df.pivot(index="method", columns="split", values=["roc_auc", "pr_auc", "f1", "accuracy"])
display(pivot)


## 9. Summary


In [ ]:
summary = {
    "task": "honest_catboost_semantic_comparison",
    "feature_sets": {
        "safe_base_features": safe_base_features,
        "popularity_features": popularity_features,
        "semantic_features": semantic_features,
        "safe_plus_popularity_features": safe_plus_popularity_features,
        "safe_plus_popularity_semantic_features": safe_plus_popularity_semantic_features,
    },
    "forbidden_exact": sorted(list(FORBIDDEN_EXACT)),
    "forbidden_patterns": FORBIDDEN_PATTERNS,
    "metrics": metrics_df.to_dict(orient="records"),
    "outputs": {
        "metrics": str(out_dir / "metrics_honest_catboost_semantic.csv"),
        "output_dir": str(out_dir)
    }
}

with open(out_dir / "honest_catboost_semantic_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

report_path = PROJECT_ROOT / "reports/experiment_reports/honest_catboost_semantic_summary.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

test_rows = metrics_df[metrics_df["split"] == "test"].copy()
report_text = "# Honest CatBoost semantic comparison\n\n"
report_text += "This report compares safe CatBoost features, train-only popularity features, and semantic embedding features.\n\n"
report_text += "Forbidden features include labels, target hours, ids, and global Steam aggregate columns.\n\n"
report_text += test_rows.to_string(index=False)
report_text += "\n"

report_path.write_text(report_text, encoding="utf-8")

print("saved metrics:", out_dir / "metrics_honest_catboost_semantic.csv")
print("saved summary:", out_dir / "honest_catboost_semantic_summary.json")
print("saved report:", report_path)
